# EDA — Give Me Some Credit

Verifies the data-quirk hypotheses listed in `docs/plans/0001-pd-scorecard-give-me-some-credit.md` §2 against the real `cs-training.csv`, per LLM.md §0.1.1 (don't clean based on hearsay — confirm against the actual file first). All outputs below are from an actual run against the real file.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.data.load import load_raw

df = load_raw('../data/raw/give-me-some-credit/cs-training.csv')
print('Shape:', df.shape)

Shape: (150000, 11)


## Target distribution

In [ ]:
print(df['SeriousDlqin2yrs'].value_counts(normalize=True))

SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64


**Finding:** 6.68% default rate — imbalanced but within the normal range for consumer credit data (not extreme). WOE binning handles this natively; no resampling needed for a scorecard.

## Missing values

In [ ]:
missing = df.isna().sum()
print(missing[missing > 0])

MonthlyIncome         29731
NumberOfDependents     3924
dtype: int64


**Finding: confirmed.** `MonthlyIncome` missing in 19.8% of rows, `NumberOfDependents` in 2.6%. Decision: treat missing as its own WOE bin for both features (industry-standard — missingness itself can carry default signal) rather than imputing.

## `age` outlier check

In [ ]:
print(df['age'].describe())
print('age==0 count:', (df['age'] == 0).sum())

count    150000.000000
mean         52.295207
std          14.771866
min           0.000000
25%          41.000000
50%          52.000000
75%          63.000000
max         109.000000
Name: age, dtype: float64
age==0 count: 1


**Finding: confirmed.** Exactly 1 row has `age == 0` — a borrower cannot have age 0, this is a data-entry error, not a real value. Decision: drop this single row before splitting (N=1, zero effect on the sample, but a nonsensical value would otherwise sit at a bin boundary).

## Past-due columns — sentinel values (96 / 98)

In [ ]:
cols = ['NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfTimes90DaysLate']
mask = df[cols].isin([96, 98]).any(axis=1)
sub = df[mask]
print('Rows with 96/98 in any past-due col:', mask.sum())
print('Identical across all 3 columns for these rows:',
      (sub[cols[0]] == sub[cols[1]]).all(), (sub[cols[1]] == sub[cols[2]]).all())
print('Default rate for these rows:', sub['SeriousDlqin2yrs'].mean())
print('Default rate overall:', df['SeriousDlqin2yrs'].mean())
print('age stats for these rows (mean, min, max):', round(sub['age'].mean(), 2), sub['age'].min(), sub['age'].max())

Rows with 96/98 in any past-due col: 269
Identical across all 3 columns for these rows: True True
Default rate for these rows: 0.5464684014869888
Default rate overall: 0.06684
age stats for these rows (mean, min, max): 34.25 21 79


**Finding: confirmed, and more informative than expected.** 269 rows (0.18%) carry the value 96 or 98 identically across all three past-due columns — these are not literal counts (96-98 late-payment events is not possible in a ~2-year window) but a sentinel/error code from the original data system. Critically, these 269 rows have a **54.6% default rate vs. 6.68% baseline (8x higher)** — this is real signal, not noise to discard.

**Decision:** treat 96/98 as a dedicated *special-value* bin in binning (not merged into the ordinary monotonic count ordering, and not dropped) — `optbinning` supports this directly via its `special_codes` parameter. Dropping these 269 rows would throw away one of the most predictive segments in the data; imputing them as an ordinary count would corrupt the monotonic ordering of the rest of the feature.

## `DebtRatio` extreme outliers — and their link to missing `MonthlyIncome`

In [ ]:
nan_income = df['MonthlyIncome'].isna()
print('DebtRatio median when MonthlyIncome is NaN:', df.loc[nan_income, 'DebtRatio'].median())
print('DebtRatio median when MonthlyIncome is present:', round(df.loc[~nan_income, 'DebtRatio'].median(), 3))
extreme = df['DebtRatio'] > 10
pct = nan_income[extreme].sum() / extreme.sum()
print(f'Rows with DebtRatio > 10: {extreme.sum()}; of those, NaN income: {nan_income[extreme].sum()} ({pct:.1%})')

DebtRatio median when MonthlyIncome is NaN: 1159.0
DebtRatio median when MonthlyIncome is present: 0.296
Rows with DebtRatio > 10: 28877; of those, NaN income: 26771 (92.7%)


**Finding: confirmed — DebtRatio and missing MonthlyIncome are entangled, not independent quirks.** When income is missing, DebtRatio's median jumps ~4000x (0.30 -> 1159) and 92.7% of the extreme-DebtRatio rows (>10) have missing income. This strongly suggests that for rows with no reported income, `DebtRatio` is not actually a ratio (likely a raw dollar debt-payment figure recorded in the same column) — a scale change, not scattered noise.

**Decision:** no special code needed here — monotonic optimal binning on the raw continuous values will naturally isolate this long tail into its own high-WOE bin regardless of the underlying cause, and `MonthlyIncome`'s own missing-bin already captures part of this signal separately. This entanglement is noted here so it isn't mistaken for a bug later; it is not treated as one.

## `RevolvingUtilizationOfUnsecuredLines` outliers

In [ ]:
print(df['RevolvingUtilizationOfUnsecuredLines'].describe(percentiles=[.5, .95, .99]))
print('RevolvingUtilization > 2 count:', (df['RevolvingUtilizationOfUnsecuredLines'] > 2).sum())

count    150000.000000
mean          6.048438
50%           0.154181
95%           1.000000
99%           1.092956
max       50708.000000
Name: RevolvingUtilizationOfUnsecuredLines, dtype: float64
RevolvingUtilization > 2 count: 371


**Finding: confirmed.** This should logically be bounded near [0,1] (a utilization ratio) but 371 rows exceed 2, up to 50,708 — clear data errors at the extreme tail. **Decision:** no clipping needed — same reasoning as `DebtRatio`: optimal monotonic binning absorbs an extreme tail into its top bin by construction, which is the actual reason this technique was chosen over feeding the raw variable into logistic regression directly (see `app/content/binning_methodology.md`, Plan 0002).

## Cleaning decisions carried into Plan 0001 §4.3 / §4.4

1. Drop the single `age == 0` row.
2. Leave `MonthlyIncome` and `NumberOfDependents` missing values as-is — binning gives them their own bin.
3. Configure `special_codes=[96, 98]` for the three past-due columns in `optbinning` — do **not** drop or impute these 269 rows.
4. Leave `DebtRatio` and `RevolvingUtilizationOfUnsecuredLines` outliers untouched (no clipping/imputation) — monotonic binning handles them by construction.
5. No resampling for class imbalance — 6.68% default rate is within the normal range for a scorecard build.